# Experimento 5 — Remoção das Variáveis Categóricas

Após os experimentos anteriores, observou-se que variáveis categóricas como:

* país (`Country`);
* tipo de corpo hídrico (`Waterbody Type`);



Essas variáveis carregam contexto ambiental importante, já que diferentes países e diferentes tipos de corpos hídricos podem apresentar características físico-químicas bastante distintas.

No entanto, também surgiu a necessidade de investigar até que ponto o modelo realmente consegue aprender padrões ambientais apenas utilizando parâmetros físico-químicos numéricos.

Diante disso, neste experimento foram removidas todas as variáveis categóricas, mantendo apenas:

* temperatura;
* ortofosfato;
* nitrogênio.

O objetivo principal deste experimento é analisar:

* como o modelo se comporta sem informações contextuais;
* quanto as variáveis categóricas influenciavam os resultados anteriores;
* se apenas variáveis numéricas conseguem manter boa capacidade preditiva;
* como o Random Forest reage ao receber menos informações de contexto ambiental.

Além disso, este experimento também ajuda a investigar se parte da performance observada anteriormente vinha realmente de relações ambientais aprendidas pelo modelo ou se estava fortemente associada a padrões específicos de países e tipos de corpos hídricos.

Outro ponto importante é que este experimento permite analisar melhor a capacidade de generalização do modelo utilizando apenas parâmetros físico-químicos.

Neste cenário, o algoritmo precisa encontrar relações ambientais mais diretas entre:

* temperatura;
* ortofosfato;
* nitrogênio;
* qualidade da água.

Sem depender de informações contextuais externas.

Neste experimento não foi aplicado balanceamento, permitindo observar inicialmente o comportamento natural do modelo apenas com variáveis numéricas e dataset desbalanceado.


## Preparação do ambiente


In [1]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
# preparando o ambiente para machine learning
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Nitrogen (mg/l)"
    ]
]

y = df["conama_status"]

In [5]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (113119, 3)
Teste: (28280, 3)


In [6]:
model = Pipeline(
    steps=[

       (
            "classifier",
            RandomForestClassifier(
                random_state=SEED
            )
        )
    ]
)

In [7]:
model.fit(X_train, y_train)

Pipeline(steps=[('classifier', RandomForestClassifier(random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [8]:
y_train_pred = model.predict(X_train)

In [10]:
train_accuracy = accuracy_score(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

Train Accuracy:
0.9235760570726403


In [11]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.9235760570726403
Train Precision:
0.9291401832501525
Train Recall:
0.9235760570726403
Train F1:
0.919174291054814
Train Confusion Matrix:
[[ 4980    63     2  2515]
 [   31 15982     2  5663]
 [    5     8   945   148]
 [   13   195     0 82567]]


In [12]:
y_pred = model.predict(X_test)

In [13]:
# Teste com 5 variáveis - sem balanceamento
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7631541725601132

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.39      0.18      0.24      1890
         Boa       0.54      0.32      0.40      5419
     Crítica       0.13      0.04      0.06       277
   Excelente       0.81      0.94      0.87     20694

    accuracy                           0.76     28280
   macro avg       0.47      0.37      0.39     28280
weighted avg       0.72      0.76      0.73     28280


Confusion Matrix:
[[  337   448    24  1081]
 [  258  1737    29  3395]
 [   46   110    11   110]
 [  234   942    21 19497]]


# Resultados Obtidos — Experimento 5

Os resultados do quinto experimento mostraram um comportamento bastante interessante do Random Forest após a remoção das variáveis categóricas.

Neste cenário, o modelo passou a utilizar apenas:

* temperatura;
* ortofosfato;
* nitrogênio.

A acurácia de treino ficou em aproximadamente 92%, enquanto a acurácia de teste ficou em torno de 76%.

Isso mostra que o modelo continuou apresentando overfitting, já que existe uma diferença considerável entre treino e teste.

Mesmo assim, a acurácia de teste permaneceu relativamente alta, indicando que as variáveis numéricas ainda carregam informações ambientais importantes para o processo de classificação.

Um ponto muito interessante observado neste experimento foi o aumento do recall da classe `Excelente`.

O modelo passou a acertar ainda mais exemplos dessa classe em comparação a vários experimentos anteriores.

Isso acontece principalmente por dois fatores:

* o dataset continua fortemente desbalanceado;
* a remoção das variáveis categóricas reduziu parte da complexidade do problema.

Sem informações contextuais como país e tipo de corpo hídrico, o modelo passou a depender muito mais dos padrões dominantes presentes nos dados numéricos.

Como a maior parte das amostras pertence à classe `Excelente`, o algoritmo naturalmente passou a favorecer ainda mais essa classe durante o aprendizado.

Isso explica por que o recall de `Excelente` aumentou significativamente.

Ao mesmo tempo, percebe-se que as classes minoritárias continuaram apresentando dificuldades de identificação.

A classe `Crítica`, por exemplo, continuou sendo extremamente difícil para o modelo, mantendo recall bastante baixo.

Isso mostra que, sem informações contextuais adicionais, o Random Forest encontra ainda mais dificuldade para separar padrões ambientais raros e complexos relacionados às situações críticas.

Outro ponto importante é que, mesmo removendo:

* `Country`;
* `Waterbody Type`;

a acurácia geral permaneceu relativamente próxima dos experimentos anteriores.

Isso sugere que:

* temperatura;
* ortofosfato;
* nitrogênio;

realmente possuem capacidade de carregar informação ambiental relevante.

Ou seja, o modelo ainda consegue aprender relações úteis utilizando apenas variáveis físico-químicas.

No entanto, a remoção das variáveis categóricas aparentemente fez o algoritmo perder parte da capacidade de contextualização ambiental.

Antes, o modelo conseguia aprender padrões específicos relacionados:

* a determinados países;
* a determinados tipos de corpos hídricos.

Agora, essas informações não existem mais, fazendo com que o algoritmo dependa apenas das relações numéricas presentes nas variáveis contínuas.

De forma geral, este experimento mostrou que:

* variáveis físico-químicas possuem forte capacidade preditiva;
* as variáveis categóricas ajudam na contextualização ambiental;
* a remoção do contexto faz o modelo favorecer ainda mais a classe `Excelente`;
* o dataset desbalanceado continua influenciando fortemente o comportamento do Random Forest;
* mesmo sem variáveis categóricas, o modelo ainda consegue manter desempenho relativamente alto.
